In [ ]:
# in databricks, I need to read csv file and append the record to the existig table name, 
# how can we achive this

# To ingest CSV files from Azure Data Lake Storage Gen2 (ADLS Gen2) into Databricks and 
# append them to an existing table

configs = {
  "fs.azure.account.auth.type": "OAuth",
  "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
  "fs.azure.account.oauth2.client.id": "<client-id>",
  "fs.azure.account.oauth2.client.secret": "<client-secret>",
  "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/<tenant-id>/oauth2/token"
}

dbutils.fs.mount(
  source = "abfss://<container>@<storage-account>.dfs.core.windows.net/",
  mount_point = "/mnt/adls",
  extra_configs = configs)



# Read CSV file into DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \   # if CSV has headers
    .option("inferSchema", "true") \
    .load("/mnt/data/new_records.csv")

# Append to existing Delta table
df.write.format("delta") 
    .option("header", "true")
    .option("mergeSchema", "true") 
    .mode("append") \
    .saveAsTable("existing_table_name")


# working with ETL pipelines and Delta tables, you can also use Auto Loader for continuous 
# ingestion if new CSV files arrive regularly:
df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("header", "true")
      .load("/mnt/data/csv_folder"))

df.writeStream.format("delta") \
    .option("checkpointLocation", "/mnt/checkpoints/csv_ingest") \
    .outputMode("append") \
    .table("existing_table_name")
